In [8]:
# ============================================================
# PLBART + DEVIGN : ONE-CELL FINAL WORKING CODE (KAGGLE)
# ============================================================

# --------------------
# Imports
# --------------------
import torch
import numpy as np
import pandas as pd
import json
from datetime import datetime

from datasets import Dataset
from transformers import (
    PLBartTokenizer,
    PLBartForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------
# Device
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# LOAD DEVIGN DATASET
# ============================================================

DATASET_PATH = "/kaggle/input/devign-dataset-plb"

def load_devign_csv(path):
    df = pd.read_csv(path)

    if "func" in df.columns:
        df["code"] = df["func"]
    elif "code" not in df.columns:
        raise ValueError("No code column found")

    if "target" in df.columns:
        df["label"] = df["target"]
    elif "label" not in df.columns:
        raise ValueError("No label column found")

    df = df[["code", "label"]]
    df["label"] = df["label"].astype(int)
    return df

print("\nLoading Devign dataset...")

train_df = load_devign_csv(f"{DATASET_PATH}/devignx_train.csv")
val_df   = load_devign_csv(f"{DATASET_PATH}/Devignx_validation.csv")
test_df  = load_devign_csv(f"{DATASET_PATH}/devignx_test.csv")

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("Label distribution:\n", train_df["label"].value_counts())

# ============================================================
# HF DATASETS
# ============================================================

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df, preserve_index=False)
test_ds  = Dataset.from_pandas(test_df, preserve_index=False)

# ============================================================
# TOKENIZER
# ============================================================

tokenizer = PLBartTokenizer.from_pretrained("uclanlp/plbart-base")

def tokenize_fn(batch):
    return tokenizer(
        batch["code"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format("torch", columns=cols)
val_ds.set_format("torch", columns=cols)
test_ds.set_format("torch", columns=cols)

# ============================================================
# MODEL
# ============================================================

model = PLBartForSequenceClassification.from_pretrained(
    "uclanlp/plbart-base",
    num_labels=2
).to(device)

# ============================================================
# TRAINING ARGUMENTS (NO CHECKPOINT SAVING)
# ============================================================

training_args = TrainingArguments(
    output_dir="./plbart_devign",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    save_strategy="no",      # 🔥 avoid disk crash
    logging_steps=100,
    report_to="none"
)

# ============================================================
# TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

# ============================================================
# TRAIN
# ============================================================

print("\nStarting PLBART training...")
trainer.train()

# ============================================================
# FINAL EVALUATION (SAFE)
# ============================================================

print("\nEvaluating on test set...")

preds = trainer.predict(test_ds)

logits = preds.predictions
if isinstance(logits, tuple):   # PLBART safety
    logits = logits[0]

y_true = preds.label_ids
y_pred = np.argmax(logits, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)

print("\n===== FINAL PLBART DEVIGN RESULTS =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("FPR      :", fpr)
print("Confusion Matrix:", tn, fp, fn, tp)
print("Prediction distribution:", np.unique(y_pred, return_counts=True))

# ============================================================
# SAVE MINIMAL RESULTS (LOW DISK SAFE)
# ============================================================

results = {
    "dataset": "Devign",
    "model": "PLBART",
    "epochs": 6,
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "fpr": float(fpr),
    "confusion_matrix": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    },
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

out_path = "/kaggle/working/PLBART_Devign_metrics.json"
with open(out_path, "w") as f:
    json.dump(results, f)

print("\nResults saved to:", out_path)


Device: cuda
GPU: Tesla T4

Loading Devign dataset...
Train: (19122, 2)
Val  : (2732, 2)
Test : (2732, 2)
Label distribution:
 label
0    10356
1     8766
Name: count, dtype: int64


sentencepiece.bpe.model:   0%|          | 0.00/986k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/783 [00:00<?, ?B/s]

Map:   0%|          | 0/19122 [00:00<?, ? examples/s]

Map:   0%|          | 0/2732 [00:00<?, ? examples/s]

...


Starting PLBART training...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,0.718700
200,0.696800
300,0.694600
400,0.690900
500,0.688200
600,0.683700
700,0.689400
800,0.690100
900,0.691500
1000,0.690000



Evaluating on test set...



===== FINAL PLBART DEVIGN RESULTS =====
Accuracy : 0.5863836017569546
Precision: 0.5575471698113208
Recall   : 0.4720447284345048
F1 Score : 0.5112456747404844
FPR      : 0.3168918918918919
Confusion Matrix: 1011 469 661 591
Prediction distribution: (array([0, 1]), array([1672, 1060]))

Results saved to: /kaggle/working/PLBART_Devign_metrics.json
